# Symmetric Lanczos — Interactive Numerical Experiments

Explore Lanczos tridiagonalization for real symmetric and complex
Hermitian matrices, investigate orthogonality and Ritz approximations,
and compare Lanczos–Galerkin with SciPy solvers.

**Instructions:** Select the parameters and click an experiment
button to display its numerical results and plots.


In [ ]:
from pathlib import Path
import sys
import time

# Locate the repository root for local Jupyter or Binder execution.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Launch Jupyter from within the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from krylov_methods.notebooks.notebook_utils import (
    spd_problem, hermitian_pd_problem, symmetric_indefinite_problem,
    residual_plot, error_plot, report,
    ritz_spectrum_plot, ritz_residual_plot,
)
from krylov_methods.symmetric_lanczos_solver.Lanczos import lanczos_tridiagonalize
from krylov_methods.symmetric_lanczos_solver.galerkin import lanczos_galerkin


def test_system(kind, n, kappa):
    """Create a reproducible Hermitian system with known exact solution."""
    if kind == "real":
        return spd_problem(n, kappa)
    if kind == "complex":
        return hermitian_pd_problem(n, kappa)
    if kind == "indefinite":
        return symmetric_indefinite_problem(n, kappa)
    raise ValueError(f"Unknown system type: {kind}")


In [ ]:
system = widgets.Dropdown(
    options=[
        ("Real SPD", "real"),
        ("Complex Hermitian PD", "complex"),
        ("Real symmetric indefinite", "indefinite"),
    ],
    value="real", description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
dimension = widgets.IntSlider(value=15, min=1, max=100, step=1, description="Krylov m")
condition = widgets.FloatLogSlider(
    value=1e3, base=10, min=1, max=6, step=0.5, description="Spectral spread"
)
reorth = widgets.Checkbox(value=True, description="Full reorthogonalization")
tol = widgets.FloatLogSlider(value=1e-10, base=10, min=-12, max=-6,
                             step=1, description="Galerkin tol")
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()
ritz_spectrum_output = widgets.Output()
ritz_residual_output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        ritz_spectrum_output.clear_output(wait=True)
        ritz_residual_output.clear_output(wait=True)
        A, b, x_true = test_system(system.value, size.value, condition.value)
        m = min(size.value, dimension.value)
        eigs = np.linalg.eigvalsh(A)
        print("SYSTEM INFORMATION")
        labels = {"real": "Real SPD", "complex": "Complex Hermitian PD",
                  "indefinite": "Real symmetric indefinite"}
        print(f"Matrix: {labels[system.value]}; dimension={size.value}; dtype={A.dtype}")
        print(f"||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Eigenvalue interval=[{eigs[0]:.3e}, {eigs[-1]:.3e}]")
        print(f"Number of negative eigenvalues={np.count_nonzero(eigs < 0)}")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Requested Krylov dimension={m}; reorthogonalize={reorth.value}")
        print(f"Galerkin stopping threshold={tol.value:.3e} (absolute)\n")

        start = time.perf_counter()
        lanczos = lanczos_tridiagonalize(A, b, m=m, reorthogonalize=reorth.value)
        time_lanczos = time.perf_counter() - start
        V, T = lanczos.basis, lanczos.tridiagonal
        orth = np.linalg.norm(V.conj().T @ V - np.eye(lanczos.iterations))
        projection = np.linalg.norm(V.conj().T @ A @ V - T)
        print("LANCZOS TRIDIAGONALIZATION")
        print(f"Achieved dimension={lanczos.iterations}; status={lanczos.message}")
        print(f"Orthogonality error ||VᴴV-I||₂={orth:.3e}")
        print(f"Projection error ||VᴴAV-T||₂={projection:.3e}")
        print(f"Tridiagonalization time={time_lanczos:.6f} s")

        # Evaluate Ritz-pair quality via the shared plotting helper.
        theta = np.linalg.eigvalsh(T)
        print("Ritz values:", np.array2string(theta, precision=5))
        ritz_residuals = np.linalg.norm(
            A @ (V @ np.linalg.eigh(T)[1]) -
            (V @ np.linalg.eigh(T)[1]) * theta, axis=0
        )
        print(f"Max Ritz-pair residual={np.max(ritz_residuals):.3e}")

        start = time.perf_counter()
        projected = lanczos_galerkin(A, b, m=m, tol=tol.value,
                                     reorthogonalize=reorth.value)
        time_galerkin = time.perf_counter() - start
        print("\nGALERKIN APPROXIMATION")
        if projected.solution is not None:
            report("Lanczos Galerkin", projected.solution, x_true, b, A,
                   lanczos.iterations, projected.converged)
            print(f"||b-Ax||₂={projected.residual_norm:.3e}; "
                  f"relative error={np.linalg.norm(projected.solution-x_true)/np.linalg.norm(x_true):.3e}")
        else:
            print("No projected solution:", projected.message)
        print(f"Galerkin call time={time_galerkin:.6f} s")

        with ritz_spectrum_output:
            ritz_spectrum_plot(A, theta)
        with ritz_residual_output:
            ritz_residual_plot(A, V, T)


button.on_click(run)


In [ ]:
study_button = widgets.Button(
    description="Explore dimension and orthogonality", button_style="info"
)
study_output = widgets.Output()
study_orth_output = widgets.Output()
study_residual_output = widgets.Output()
study_error_output = widgets.Output()


def run_study(_):
    with study_output:
        clear_output(wait=True)
        study_orth_output.clear_output(wait=True)
        study_residual_output.clear_output(wait=True)
        study_error_output.clear_output(wait=True)
        A, b, x_true = test_system(system.value, size.value, condition.value)
        # A few Krylov dimensions keep this interactive experiment responsive.
        candidates = sorted(set([
            1, min(3, size.value), min(5, size.value),
            min(10, size.value), min(15, size.value),
            min(20, size.value), size.value,
        ]))
        results = {False: {"orth": [], "res": [], "error": []},
                   True: {"orth": [], "res": [], "error": []}}
        for use_reorth in (False, True):
            for requested in candidates:
                L = lanczos_tridiagonalize(A, b, m=requested,
                                          reorthogonalize=use_reorth)
                Q = L.basis
                orth_err = np.linalg.norm(Q.conj().T @ Q - np.eye(L.iterations))
                projected = lanczos_galerkin(A, b, m=requested, tol=1e-12,
                                             reorthogonalize=use_reorth)
                results[use_reorth]["orth"].append(orth_err)
                results[use_reorth]["res"].append(
                    np.nan if projected.residual_norm is None else projected.residual_norm
                )
                results[use_reorth]["error"].append(
                    np.nan if projected.solution is None else
                    np.linalg.norm(projected.solution-x_true)/np.linalg.norm(x_true)
                )
        # Plot the three metrics independently.
        study_metrics = (
            (
                "orth",
                r"$\|V_m^H V_m-I\|_2$",
                "Lanczos Basis Orthogonality",
                study_orth_output,
            ),
            (
                "res",
                r"$\|b-Ax_m\|_2$",
                "Galerkin Residual",
                study_residual_output,
            ),
            (
                "error",
                r"$\|x_m-x^*\|_2/\|x^*\|_2$",
                "Relative Solution Error",
                study_error_output,
            ),
        )

        for key, ylabel, title, plot_output in study_metrics:
            with plot_output:
                fig, ax = plt.subplots(figsize=(4.1, 3.0))

                for use_reorth, color in (
                    (False, "tab:orange"),
                    (True, "tab:blue"),
                ):
                    label = (
                        "Full reorthogonalization"
                        if use_reorth
                        else "Three-term only"
                    )

                    y = np.asarray(
                        results[use_reorth][key],
                        dtype=float,
                    )

                    ax.semilogy(
                        candidates,
                        np.where(
                            np.isfinite(y),
                            np.maximum(
                                y,
                                np.finfo(float).tiny,
                            ),
                            np.nan,
                        ),
                        marker="o",
                        label=label,
                        color=color,
                    )

                ax.set(
                    xlabel="Requested Krylov dimension m",
                    ylabel=ylabel,
                    title=title,
                )
                ax.grid(True, which="both", alpha=0.25)
                ax.margins(y=0.12)
                ax.legend(fontsize=7)
                fig.tight_layout()
                plt.show()
        print("NaN or missing curve points indicate an undefined singular projected solve.")
        print("Full reorthogonalization can improve orthogonality, at extra cost.")


study_button.on_click(run_study)


In [ ]:
from scipy.sparse.linalg import cg as scipy_cg, minres as scipy_minres

solve_button = widgets.Button(description="Solve Ax=b and compare", button_style="success")
solve_output = widgets.Output()
solve_residual_output = widgets.Output()
solve_error_output = widgets.Output()


def solve_system(_):
    with solve_output:
        clear_output(wait=True)
        solve_residual_output.clear_output(wait=True)
        solve_error_output.clear_output(wait=True)
        A, b, x_true = test_system(system.value, size.value, condition.value)
        n = A.shape[0]
        selected_m = min(dimension.value, n)
        x_direct = np.linalg.solve(A, b)
        exact_err = np.linalg.norm(x_direct - x_true) / np.linalg.norm(x_true)
        print("LINEAR SYSTEM")
        print(f"System={system.value}, dimension={n}, cond₂(A)={np.linalg.cond(A):.3e}")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Direct solve relative error={exact_err:.3e}")
        print(f"Selected Krylov dimension m={selected_m}; absolute Galerkin tolerance={tol.value:.1e}\n")

        # The projected method is tested both at the selected dimension and
        # at the full dimension to illustrate finite-dimensional convergence.
        for label, m in (("Selected dimension", selected_m), ("Full dimension", n)):
            start = time.perf_counter()
            result = lanczos_galerkin(A, b, m=m, tol=tol.value,
                                      reorthogonalize=reorth.value)
            elapsed = time.perf_counter() - start
            if result.solution is None:
                print(f"Lanczos–Galerkin ({label}, m={m}): {result.message}")
            else:
                report(f"Galerkin m={m}", result.solution, x_true, b, A,
                       result.lanczos.iterations, result.converged)
                print(f"  Time={elapsed:.6f} s; {result.message}")

        # SciPy's relative tolerance is adjusted to roughly match the
        # absolute true-residual threshold used by Lanczos–Galerkin.
        scipy_rtol = tol.value / max(np.linalg.norm(b), np.finfo(float).tiny)
        scipy_history = [np.linalg.norm(b)]
        def cb(xk):
            scipy_history.append(np.linalg.norm(b - A @ xk))
        start = time.perf_counter()
        if system.value == "indefinite":
            scipy_x, info = scipy_minres(A, b, rtol=scipy_rtol,
                                        maxiter=max(5*n, 100), callback=cb)
            scipy_label = "SciPy MINRES"
        else:
            scipy_x, info = scipy_cg(A, b, rtol=scipy_rtol, atol=0.0,
                                     maxiter=max(5*n, 100), callback=cb)
            scipy_label = "SciPy CG"
        elapsed = time.perf_counter() - start
        report(scipy_label, scipy_x, x_true, b, A,
               len(scipy_history)-1, info == 0)
        print(f"  Time={elapsed:.6f} s; SciPy info={info}\n")

        # Sample at most ~14 dimensions to keep interactive use responsive.
        sample = sorted(set([1, selected_m, n] +
                            list(range(1, n+1, max(1, n // 12)))))
        dims, residual_norms, relative_errors = [], [], []
        for m in sample:
            result = lanczos_galerkin(A, b, m=m, tol=tol.value,
                                      reorthogonalize=reorth.value)
            dims.append(m)
            if result.solution is None:
                residual_norms.append(np.nan)
                relative_errors.append(np.nan)
            else:
                residual_norms.append(np.linalg.norm(b-A@result.solution))
                relative_errors.append(
                    np.linalg.norm(result.solution-x_true)/np.linalg.norm(x_true)
                )
        dims = np.asarray(dims)
        residual_norms = np.asarray(residual_norms)
        relative_errors = np.asarray(relative_errors)
        # The shared plot helpers reject NaN histories; show undefined points
        # as gaps on the axes while preserving their true m coordinates.
        def plot_finite(ax, values, label):
            good = np.isfinite(values)
            ax.semilogy(dims[good], np.maximum(values[good], np.finfo(float).tiny),
                        marker="o", linewidth=1.5, label=label)

        # True residual versus Krylov dimension.
        with solve_residual_output:
            fig, ax = plt.subplots(figsize=(4.1, 3.0))

            plot_finite(
                ax,
                residual_norms,
                "Lanczos–Galerkin",
            )

            ax.axhline(
                tol.value,
                color="black",
                linestyle="--",
                label="Galerkin tolerance",
            )

            ax.set(
                xlabel="Krylov dimension m",
                ylabel=r"True residual $\|b-Ax_m\|_2$",
                title="Residual versus Krylov dimension",
            )

            ax.grid(True, which="both", alpha=0.25)
            ax.legend(fontsize=7)
            ax.margins(y=0.12)

            fig.tight_layout()
            plt.show()

        # Relative solution error versus Krylov dimension.
        with solve_error_output:
            fig, ax = plt.subplots(figsize=(4.1, 3.0))

            plot_finite(
                ax,
                relative_errors,
                "Lanczos–Galerkin",
            )

            ax.set(
                xlabel="Krylov dimension m",
                ylabel=r"Relative error $\|x_m-x^*\|_2/\|x^*\|_2$",
                title="Forward error versus Krylov dimension",
            )

            ax.grid(True, which="both", alpha=0.25)
            ax.legend(fontsize=7)
            ax.margins(y=0.12)

            fig.tight_layout()
            plt.show()
        if np.any(~np.isfinite(residual_norms)):
            print("Some Krylov dimensions produced singular projected systems; "
                  "their plot points are omitted.")
        print("Interpretation: compare the true residual and relative solution error; "
              "the relationship depends on the conditioning of A.")


solve_button.on_click(solve_system)


In [ ]:

# -------------------------------------------------------
# Symmetric Lanczos — Compact Voilà dashboard
# -------------------------------------------------------

lanczos_status = widgets.HTML()
study_status = widgets.HTML()
solve_status = widgets.HTML()

lanczos_parameters = (
    system, size, dimension, condition, reorth, tol
)

study_parameters = (
    system, size, condition
)

solve_parameters = (
    system, size, dimension, condition, reorth, tol
)

last_runs = {
    "lanczos": None,
    "study": None,
    "solve": None,
}


# -------------------------------------------------------
# Parameter-change indicators
# -------------------------------------------------------

def parameter_values(parameters):
    return tuple(widget.value for widget in parameters)


def update_status(name, parameters, indicator):
    previous = last_runs[name]

    if previous is None:
        indicator.value = ""
    elif parameter_values(parameters) == previous:
        indicator.value = (
            "<span style='color:#22863a;font-weight:600;'>"
            "● Simulation up to date</span>"
        )
    else:
        indicator.value = (
            "<span style='color:#b36b00;font-weight:600;'>"
            "● Parameters changed — rerun required</span>"
        )


def refresh_lanczos(change=None):
    update_status(
        "lanczos", lanczos_parameters, lanczos_status
    )


def refresh_study(change=None):
    update_status(
        "study", study_parameters, study_status
    )


def refresh_solve(change=None):
    update_status(
        "solve", solve_parameters, solve_status
    )


for parameter in lanczos_parameters:
    parameter.observe(refresh_lanczos, names="value")

for parameter in study_parameters:
    parameter.observe(refresh_study, names="value")

for parameter in solve_parameters:
    parameter.observe(refresh_solve, names="value")


# -------------------------------------------------------
# Seven independent plot panels
# -------------------------------------------------------

def plot_panel(title, plot_output):
    return widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-weight:600;"
                "text-align:center;font-size:13px;'>"
                + title
                + "</div>"
            ),
            plot_output,
        ],
        layout=widgets.Layout(
            flex="1 1 240px",
            min_width="220px",
            max_width="100%",
            padding="4px",
            overflow="hidden",
        ),
    )


ritz_spectrum_panel = plot_panel(
    "Ritz Spectrum",
    ritz_spectrum_output,
)

ritz_residual_panel = plot_panel(
    "Ritz-Pair Residuals",
    ritz_residual_output,
)

study_orth_panel = plot_panel(
    "Orthogonality Error",
    study_orth_output,
)

study_residual_panel = plot_panel(
    "Galerkin Residual",
    study_residual_output,
)

study_error_panel = plot_panel(
    "Relative Solution Error",
    study_error_output,
)

solve_residual_panel = plot_panel(
    "Galerkin Solver Residual",
    solve_residual_output,
)

solve_error_panel = plot_panel(
    "Galerkin Solver Forward Error",
    solve_error_output,
)

lanczos_plot_panels = (
    ritz_spectrum_panel,
    ritz_residual_panel,
)

study_plot_panels = (
    study_orth_panel,
    study_residual_panel,
    study_error_panel,
)

solve_plot_panels = (
    solve_residual_panel,
    solve_error_panel,
)

all_plot_panels = (
    *lanczos_plot_panels,
    *study_plot_panels,
    *solve_plot_panels,
)

for panel in all_plot_panels:
    panel.layout.display = "none"

plots_row = widgets.HBox(
    list(all_plot_panels),
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="5px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Expandable numerical reports
# -------------------------------------------------------

lanczos_details = widgets.Accordion(children=[output])
lanczos_details.set_title(
    0, "Lanczos tridiagonalization & Ritz diagnostics"
)
lanczos_details.selected_index = None

study_details = widgets.Accordion(
    children=[study_output]
)
study_details.set_title(
    0, "Dimension & orthogonality numerical details"
)
study_details.selected_index = None

solve_details = widgets.Accordion(
    children=[solve_output]
)
solve_details.set_title(
    0, "Lanczos–Galerkin & SciPy solver comparison"
)
solve_details.selected_index = None

for details in (
    lanczos_details,
    study_details,
    solve_details,
):
    details.layout.display = "none"
    details.layout.flex = "1 1 300px"
    details.layout.min_width = "240px"


# -------------------------------------------------------
# Reveal results after running an experiment
# -------------------------------------------------------

def show_lanczos_results(change=None):
    for panel in lanczos_plot_panels:
        panel.layout.display = ""

    lanczos_details.layout.display = ""

    last_runs["lanczos"] = parameter_values(
        lanczos_parameters
    )
    refresh_lanczos()


def show_study_results(change=None):
    for panel in study_plot_panels:
        panel.layout.display = ""

    study_details.layout.display = ""

    last_runs["study"] = parameter_values(
        study_parameters
    )
    refresh_study()


def show_solve_results(change=None):
    for panel in solve_plot_panels:
        panel.layout.display = ""

    solve_details.layout.display = ""

    last_runs["solve"] = parameter_values(
        solve_parameters
    )
    refresh_solve()


# Register presentation callbacks after numerical callbacks.
button.on_click(show_lanczos_results)
study_button.on_click(show_study_results)
solve_button.on_click(show_solve_results)


# -------------------------------------------------------
# Experiment control panels
# -------------------------------------------------------

lanczos_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Lanczos Tridiagonalization</h3>"
        ),
        system,
        size,
        dimension,
        condition,
        reorth,
        tol,
        button,
        lanczos_status,
    ],
    layout=widgets.Layout(
        flex="2 1 440px",
        min_width="300px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

study_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Dimension & Orthogonality</h3>"
        ),
        widgets.HTML(
            "<p style='font-size:13px;'>"
            "Compare Krylov dimensions with and without "
            "full reorthogonalization."
            "</p>"
        ),
        widgets.HTML(
            "<p style='font-size:13px;'>"
            "Uses the system, matrix size, and spectral "
            "spread from the first panel."
            "</p>"
        ),
        study_button,
        study_status,
    ],
    layout=widgets.Layout(
        flex="1 1 270px",
        min_width="240px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

solve_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3>Lanczos–Galerkin Solver</h3>"
        ),
        widgets.HTML(
            "<p style='font-size:13px;'>"
            "Solve Ax=b and compare Lanczos–Galerkin "
            "with SciPy CG or MINRES."
            "</p>"
        ),
        widgets.HTML(
            "<p style='font-size:13px;'>"
            "Uses all parameters from the first panel."
            "</p>"
        ),
        solve_button,
        solve_status,
    ],
    layout=widgets.Layout(
        flex="1 1 270px",
        min_width="240px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

controls_row = widgets.HBox(
    [
        lanczos_controls,
        study_controls,
        solve_controls,
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="stretch",
    ),
)

details_row = widgets.HBox(
    [
        lanczos_details,
        study_details,
        solve_details,
    ],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Final interactive dashboard
# -------------------------------------------------------

lanczos_dashboard = widgets.VBox(
    [
        widgets.HTML("""
        <style>
        .lanczos-compact-dashboard .output_area img,
        .lanczos-compact-dashboard .jp-OutputArea-output img {
            max-width:100% !important;
            height:auto !important;
        }
        </style>
        """),
        controls_row,
        plots_row,
        details_row,
    ],
    layout=widgets.Layout(
        width="100%",
        gap="12px",
    ),
)

lanczos_dashboard.add_class(
    "lanczos-compact-dashboard"
)

display(lanczos_dashboard)

# No automatic simulations.
# All plots and reports remain hidden until requested.
